# FramePipe: reproduce the GPU results

Builds FramePipe with CUDA on a free Kaggle (or Colab) GPU, runs the GPU tests, the independent NumPy oracle, Compute Sanitizer, and a short benchmark, then draws the charts.

**Kaggle:** Settings → Accelerator → *GPU T4 x2*, and Settings → Internet → *On*. **Colab:** Runtime → Change runtime type → *T4 GPU*.

Total time is about 10 minutes, most of it building OpenCV from pinned, SHA-256-verified sources.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!nvcc --version | tail -2

In [ ]:
import os
workdir = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
%cd {workdir}
!test -d framepipe || git clone --depth 1 https://github.com/shres1974/framepipe.git
%cd framepipe
!pip install -q -r requirements.txt
os.environ['FRAMEPIPE_DEPS'] = os.path.expanduser('~/.local/framepipe-deps')
!git log -1 --oneline

In [ ]:
# Pinned C++ dependencies (OpenCV 4.10, GoogleTest 1.15.2, nlohmann/json 3.11.3), ~6 min
!scripts/install_deps.sh > deps.log 2>&1 && echo deps ok || tail -40 deps.log

In [ ]:
!cmake --preset cuda > configure.log 2>&1 && cmake --build --preset cuda -j $(nproc) > build.log 2>&1 && echo build ok || tail -40 build.log
!build/cuda/bin/framepipe doctor

## Correctness

GPU hardware tests (one test is expected to skip: it only applies when *no* device is present), the full suite, and the independent NumPy oracle with `cuda` as a candidate.

In [ ]:
!build/cuda/bin/framepipe_cuda_tests --gtest_brief=1
!ctest --test-dir build/cuda --output-on-failure | tail -5
!python3 scripts/run_oracle_suite.py --framepipe build/cuda/bin/framepipe --work-dir build/cuda/ow --candidates cpu-ref,cpu-omp,cuda 2>&1 | tail -3

In [ ]:
# Compute Sanitizer: memory errors, races, uninitialized reads, sync misuse (~3 min)
!for t in memcheck racecheck initcheck synccheck; do echo "== $t"; compute-sanitizer --tool $t --error-exitcode 99 build/cuda/bin/framepipe_cuda_tests --gtest_brief=1 2>&1 | tail -2; done

## Benchmark

A short run (fewer iterations than the committed evidence). Every backend is parity-checked against `cpu-ref` before it is timed.

In [ ]:
!python3 scripts/generate_fixtures.py benchmark --out data/bench > /dev/null
!WORKLOADS=repeat WARMUP=10 ITERATIONS=50 REPETITIONS=2 benchmarks/run_matrix.sh build/cuda results/local/notebook cpu-ref,cpu-omp,opencv-cpu,cuda $(nproc) > bench.log 2>&1 && echo bench ok || tail -40 bench.log
!for d in results/local/notebook/*/; do python3 scripts/validate_results.py $d > /dev/null || echo INVALID $d; done; echo validated

In [ ]:
!python3 scripts/make_showcase.py all --framepipe build/cuda/bin/framepipe --evidence results/local/notebook
from IPython.display import Image, display
for f in ['hero', 'core_latency', 'gpu_breakdown', 'end_to_end']:
    display(Image(f'docs/assets/{f}.png'))